# MCP (Model Context Protocol) Server Implementation

In [1]:
!python -m pip install fastmcp langchain_mcp_adapters --quiet

In [2]:
import os
os.makedirs("mcpserver",exist_ok=True)

### MCP Server Implementation: Streamable http mode

In [3]:
%%writefile mcpserver/mcpserver1.py

from mcp.server.fastmcp import FastMCP
import requests, json
import wikipedia

mcp = FastMCP("TredenceMCP")

@mcp.tool()
async def get_current_weather(city:str)->dict:
    """ this funciton can be used to get current weather information for any given city using openweathermap"""
    api_key="6a8b0ac166a37e2b7a38e64416b3c3fe"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = json.loads(response.content.decode())
    output = {"city":city,"weather":response['weather'][0]['description'],
              "temperature":response['main']['temp'], "unit":"kelvin"
              }
    return output

@mcp.tool()
async def get_wikipedia_summary(query:str)->str:
    response = wikipedia.summary(query)
    return response


if __name__=="__main__":
    mcp.run(transport='streamable-http')


Writing mcpserver/mcpserver1.py


In [9]:
# run mCP server: python mcpserver/mcpserver1.py

## Implement an Agent which connects to the MCP server and fetches the tools

In [4]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

In [5]:
client = MultiServerMCPClient({"TredenceMCP":{"url":"http://127.0.0.1:8000/mcp",
                                              "transport":"streamable_http"}})


tools = await client.get_tools()
tools

[StructuredTool(name='get_current_weather', description=' this funciton can be used to get current weather information for any given city using openweathermap', args_schema={'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_current_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x79d7bc0e68e0>),
 StructuredTool(name='get_wikipedia_summary', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'get_wikipedia_summaryArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x79d7bbf48c20>)]

In [6]:
agent = create_agent("google_genai:gemini-2.5-flash",tools)
await agent.ainvoke({"messages":[{"role":'user',"content":"what is the weather in Delhi?"}]})

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


{'messages': [HumanMessage(content='what is the weather in Delhi?', additional_kwargs={}, response_metadata={}, id='f8af0202-69e1-41ae-801e-535b844dd7c2'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_weather', 'arguments': '{"city": "Delhi"}'}, '__gemini_function_call_thought_signatures__': {'352421c2-e018-4834-844f-57470e533cfb': 'CvYBAQw51sdDN9E4ZCj/N3Vp1PNSL19tP4AAuXA9qjeqaC7Jvr/crG4jlfFprkt8redIhrBvfaIsVP/XqjV0pxgk+lldZ9edjn3sWdf3wMFu9EMxrsW7xwOP9woSba2+b10NKL6qpm3p54QQWW7IhRGbUebaEfblRodNLmSf9iFef0tqbx7SK+jypUTrIh6dCWoYZ5bh3fYoW2ejR/e40DBTejig7Sgo5ywmWZTDw70Xc9G34VSwQnmISLeretv7GRbbjXSl51+8duPhbaM/TBlRy+f3KU3KjrpPRwMhcoHaDXL6tC8lKamK6lYWMgnkGWr/WlLKtdlK'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e1556-3b6d-7430-8239-658027f98dd4-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Delhi'}, 'id': '352421c2-e018-4834-844f-5

In [7]:
await agent.ainvoke({"messages":[{"role":'user',"content":"Tell me more about city Jaiselmer?"}]})

{'messages': [HumanMessage(content='Tell me more about city Jaiselmer?', additional_kwargs={}, response_metadata={}, id='69666576-ef9e-405a-ac26-27a94da70fd1'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_wikipedia_summary', 'arguments': '{"query": "Jaiselmer"}'}, '__gemini_function_call_thought_signatures__': {'8c732f4e-f8ac-49df-b2e9-c829051417d7': 'CskCAQw51se+lCQCVj6oXk6UpYJzyOuVvKUiz4PcVj2bhuWHvNoQcgtKOabWiUTuMeAfKfpKyYfbILewtRkK7u1k6J4UxAZO6DlhtM7Ovu9C2iYPKN+koxkdq/B5jyNIHkGN1TK06oF84Jc0f+n0QOmBwgbR1E+CFmqH71PIy3DmLzuSkQVIhaR1BVNPJVGZQWbBzXVxs0kA9tzjL2XYw9rgalCsA8g+voaFG3OwFg50WuFS3v0tSXBkeJ8St3XjWJs7Pa0fOVq39eZEWEBFf0uzhIJH+9xZoPi1ma+uuhx5KlcUq/1Oi9VZ3Zv/72AYTvZDyMJDH1dO5n6uxo6qS0TPtQsP6QFDpIU+Hm9SWlNLBS6nyufXCzz3yacYTa8lF6vAwyfMAk+j4jLMmSBC+faUd+z089IsKZqKG5gC0Hmoa+r9BEI7TJC5VqU='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e1556-a3ba-70c3-8

### MCP Server Implementation: stdio
 mode

In [8]:
%%writefile mcpserver/mcpserver2.py

from mcp.server.fastmcp import FastMCP
import requests, json
import wikipedia

mcp = FastMCP("TredenceMCP")

@mcp.tool()
def get_current_weather(city:str)->dict:
    """ this funciton can be used to get current weather information"""
    api_key="6a8b0ac166a37e2b7a38e64416b3c3fe"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = json.loads(response.content.decode())
    output = {"city":city,"weather":response['weather'][0]['description'],
              "temperature":response['main']['temp'], "unit":"kelvin"
              }
    return output

@mcp.tool()
def get_wikipedia_summary(query:str)->str:
    response = wikipedia.summary(query)
    return response


if __name__=="__main__":
    mcp.run(transport='stdio')


Writing mcpserver/mcpserver2.py


In [9]:
client = MultiServerMCPClient({"TredenceMCP":{"command":"python","args":["mcpserver/mcpserver2.py"],
                                              "transport":"stdio"}})


tools = await client.get_tools()
tools

[StructuredTool(name='get_current_weather', description=' this funciton can be used to get current weather information', args_schema={'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_current_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x79d7bbf49120>),
 StructuredTool(name='get_wikipedia_summary', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'get_wikipedia_summaryArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x79d7ba37f380>)]

In [10]:
agent = create_agent("google_genai:gemini-2.5-flash",tools)
await agent.ainvoke({"messages":[{"role":'user',"content":"what is the weather in Delhi?"}]})

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


{'messages': [HumanMessage(content='what is the weather in Delhi?', additional_kwargs={}, response_metadata={}, id='408f1ea7-9878-4bbe-99e1-be97793738ee'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_weather', 'arguments': '{"city": "Delhi"}'}, '__gemini_function_call_thought_signatures__': {'2c1530f1-cda1-409c-8a62-bfc36a8b3c6b': 'CtkBAQw51se5ycT3vEUBYhodZNmTPZMFgWMW25f5YyY/8zROjZ44whsEgk3mZjAFAL+RkrCMxXNTlBoXTsPi7i+KJjTWp5zOxkPKbjTwDF4til6okoeeKTdtA6ItufsjcuLnAuvAaq6N79FzVb0S7nflcWpuZLtbKJY4m9DFE07A9zIMYuYbVZWqR3CmZpaSlxWE6wy6jhcKeTNiFwJ5Gyk9sXnh/CzOS9ZRoLtBoepjm3YtWRUP3wn2ItpjeVjVi6EGz4WVTu9TO4RM8j3HT9QP64AeaX5mvR5rOA=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e1559-f9a1-7600-aaca-073075bd8cf7-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Delhi'}, 'id': '2c1530f1-cda1-409c-8a62-bfc36a8b3c6b', 'type': 'tool_call'}],